In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import f_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
from matplotlib import pyplot as plt

In [ ]:
dataset = pd.read_csv("../data/train.csv")
independent_variables = dataset.iloc[:, :-1]
dependent_variable = dataset.iloc[:,-1]
seed=42

In [ ]:
# Scale the data
def z_score_scale(dataset, feature_name):
    feature = list(dataset[feature_name])
    mu = np.mean(feature)
    std = np.std(feature)
    z_score = [round((i - mu) / std, 2) for i in feature]

    dataset[feature_name] = z_score

features_to_scale = ["wheelbase", "carlength", "carwidth", "carheight", "curbweight", "enginesize", "compressionratio", "horsepower", "peakrpm", "highwaympg", "citympg"]
# To avoid data leakage, we first split the data into train and test sets. I will use 90% train, and 20% test set.
X_train, X_test, y_train, y_test = train_test_split(independent_variables, dependent_variable, random_state=seed, test_size=0.2, shuffle=True)

# Scale train
for i in X_train:
    if i in features_to_scale:
        z_score_scale(X_train, i)

# Scale test
for i in X_test:
    if i in features_to_scale:
        z_score_scale(X_test, i)


In [ ]:
# Feature selection
# When we have many features, we need to check which feature is important for our target. 
# Here I use F regression and check P values which tells us how important the feature is. 
p_values = {}
for f in independent_variables:
    p_value = f_regression(dataset[f].values.reshape(-1, 1), dependent_variable)[1]
    p_values[f] = p_value

In [ ]:
p_values

In [ ]:
# Helper functions
def calculate_metrics(y_test, preds):
    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = root_mean_squared_error(y_test, preds)
    r2 = r2_score(y_test, preds)


    print(f"Mean Absolute Error (MAE): {mae:.2f}")
    print(f"Mean Squared Error (MSE): {mse:.2f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
    print(f"R-squared (R2) Score: {r2:.2f}")

    return (mae, mse, rmse, r2)
    
def plot_pred_vs_true(y_test, preds):
    MIN_VALUE = 0
    MAX_VALUE = max(y_test)
    plt.figure(figsize=(8, 8))
    plt.scatter(y_test, preds)
    plt.plot([MIN_VALUE, MAX_VALUE], [MIN_VALUE, MAX_VALUE], '--k', label="Correct prediction")

    plt.title("Predicted vs. True Plot")
    plt.xlabel('True Price')
    plt.ylabel('Predicted Price')
    plt.legend()
    plt.tight_layout()

def plot_residual(y_test, preds):
    residuals = y_test - preds
    plt.figure(figsize=(8, 8))
    plt.xlabel('Price')
    plt.ylabel('Residuals')
    plt.title('Residual Plot')
    plt.scatter(preds, residuals)
    plt.axhline(0, linestyle="--")

In [ ]:
simple_linear_model = LinearRegression()
simple_linear_model.fit(X_train, y_train)
simple_linear_model_preds = simple_linear_model.predict(X_test)
simple_linear_model_preds_on_train = simple_linear_model.predict(X_train)

print("Train set performance: \n")
mae, mse, rmse, r2 = calculate_metrics(y_test=y_train, preds=simple_linear_model_preds_on_train)
print("\nTest set performance: \n")
calculate_metrics(y_test=y_test, preds=simple_linear_model_preds)


In [ ]:
plot_pred_vs_true(y_test, simple_linear_model_preds)

In [ ]:
plot_residual(y_test, simple_linear_model_preds)

In [ ]:
# This tells us heteroskedasticity -> the plot shows us a spread / unequal variance.

In [ ]:
# We could help on this with Box-Cox transformation
# https://datascienceplus.com/how-to-detect-heteroscedasticity-and-rectify-it/
# https://medium.com/@lomashbhuva/mastering-data-transformations-a-deep-dive-into-box-cox-and-yeo-johnson-transformations-1beb17737196
# Strictly positive data: Box-Cox requires all input values to be greater than zero. So I choosed yeo-johnson which handles 0 and negative values.
from sklearn.preprocessing import power_transform

X_train, X_test, y_train, y_test = train_test_split(independent_variables, dependent_variable, random_state=seed, test_size=0.2, shuffle=True)

X_train = power_transform(X_train, method='yeo-johnson', standardize=True, copy=True)
X_test = power_transform(X_test, method='yeo-johnson', standardize=True, copy=True)

In [ ]:
pwr_linear_model_preds = LinearRegression()
pwr_linear_model_preds.fit(X_train, y_train)

pwr_preds_on_train = pwr_linear_model_preds.predict(X_train)
pwr_preds = pwr_linear_model_preds.predict(X_test)
print("Train set performance: \n")
calculate_metrics(y_test=y_train, preds=pwr_preds_on_train)
print("\nTest set performance: \n")
calculate_metrics(y_test=y_test, preds=pwr_preds)

In [ ]:
plot_pred_vs_true(y_test, pwr_preds)

In [ ]:
plot_residual(y_test, pwr_preds)

In [ ]:
# I'm considering to use regression
from sklearn.linear_model import Ridge
ridge_model = Ridge(alpha=.5)
ridge_model.fit(X_train, y_train)
ridge_preds = ridge_model.predict(X_test)
ridge_preds_on_train = ridge_model.predict(X_train)
print("Train set performance: \n")
calculate_metrics(y_test=y_train, preds=ridge_preds_on_train)
print("\nTest set performance: \n")
calculate_metrics(y_test=y_test, preds=ridge_preds)

In [ ]:
plot_pred_vs_true(y_test, ridge_preds)

In [ ]:
plot_residual(y_test, ridge_preds)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

random_frst_reg = RandomForestRegressor(
    n_estimators=5,
    random_state=seed,
)

random_frst_reg.fit(X_train, y_train)
rnd_frst_preds = random_frst_reg.predict(X_test)
rnd_frst_preds_on_train = random_frst_reg.predict(X_train)
print("Train set performance: \n")
calculate_metrics(y_test=y_train, preds=rnd_frst_preds_on_train)
print("\nTest set performance: \n")
calculate_metrics(y_test=y_test, preds=rnd_frst_preds)

In [ ]:
# Currently the simplest linear regression model is the best. So I would like to improve that, this will be my baseline.
# I'm thinking that I add car dimension as a next feature which is (hegiht * width * length)

features_to_scale = ["wheelbase", "carlength", "carwidth", "carheight", "curbweight", "enginesize", "compressionratio", "horsepower", "peakrpm", "highwaympg", "citympg", "dimension"]

independent_variables["dimension"] = independent_variables["carlength"] * independent_variables["carheight"] * independent_variables["carwidth"]

X_train, X_test, y_train, y_test = train_test_split(independent_variables, dependent_variable, random_state=seed, test_size=0.2, shuffle=True)

# Scale train
for i in X_train:
    if i in features_to_scale:
        z_score_scale(X_train, i)

# Scale test
for i in X_test:
    if i in features_to_scale:
        z_score_scale(X_test, i)

In [ ]:
simple_linear_model = LinearRegression()
simple_linear_model.fit(X_train, y_train)
simple_linear_model_preds = simple_linear_model.predict(X_test)
simple_linear_model_preds_on_train = simple_linear_model.predict(X_train)

print("Train set performance: \n")
mae, mse, rmse, r2 = calculate_metrics(y_test=y_train, preds=simple_linear_model_preds_on_train)
print("\nTest set performance: \n")
calculate_metrics(y_test=y_test, preds=simple_linear_model_preds)

In [ ]:
# Okay, this helped a little. Based on P values, remove those which are not so important, this is my next step.

features_to_scale = ["wheelbase", "carlength", "carwidth", "carheight", "curbweight", "enginesize", "compressionratio", "horsepower", "peakrpm", "highwaympg", "citympg", "dimension"]

independent_variables["dimension"] = independent_variables["carlength"] * independent_variables["carheight"] * independent_variables["carwidth"]

independent_variables = independent_variables.drop(["carlength", "carheight", "carwidth"], axis=1)

# and also drop above a threshold
threshold = 0.1
for feature in p_values.keys():
    print(feature)
    if float(p_values[feature][0]) > threshold:
        independent_variables = independent_variables.drop([feature], axis=1)


X_train, X_test, y_train, y_test = train_test_split(independent_variables, dependent_variable, random_state=seed, test_size=0.2, shuffle=True)

# Scale train
for i in X_train:
    if i in features_to_scale:
        z_score_scale(X_train, i)

# Scale test
for i in X_test:
    if i in features_to_scale:
        z_score_scale(X_test, i)


In [ ]:
independent_variables.describe()

In [ ]:
simple_linear_model = LinearRegression()
simple_linear_model.fit(X_train, y_train)
simple_linear_model_preds = simple_linear_model.predict(X_test)
simple_linear_model_preds_on_train = simple_linear_model.predict(X_train)

print("Train set performance: \n")
mae, mse, rmse, r2 = calculate_metrics(y_test=y_train, preds=simple_linear_model_preds_on_train)
print("\nTest set performance: \n")
calculate_metrics(y_test=y_test, preds=simple_linear_model_preds)